In [1]:
from txgnn import TxData, TxGNN, TxEval

TxData = TxData(data_folder_path = 'data/kg')
TxData.prepare_split(split = 'complex_disease', seed = 42, no_kg = False)

Found local copy...
Found local copy...
Found local copy...
Found saved processed KG... Loading...
Splits detected... Loading splits....
Creating DGL graph....
Done!


In [2]:
TxGNN = TxGNN(data = TxData, 
              weight_bias_track = False,
              proj_name = 'TxGNN',
              exp_name = 'TxGNN',
              device='cpu'
              )

# to load a pretrained model: 
# TxGNN.load_pretrained('./model_ckpt')

TxGNN.model_initialize(n_hid = 100, 
                      n_inp = 100, 
                      n_out = 100, 
                      proto = True,
                      proto_num = 3,
                      attention = False,
                      sim_measure = 'all_nodes_profile',
                      bert_measure = 'disease_name',
                      agg_measure = 'rarity',
                      num_walks = 200,
                      walk_mode = 'bit',
                      path_length = 2)

In [ ]:
## here we did not run this, since the output is too long to fit into the notebook
TxGNN.pretrain(n_epoch = 2, 
               learning_rate = 1e-3,
               batch_size = 1024, 
               train_print_per_n = 20)

Creating minibatch pretraining dataloader...
Start pre-training with #param: 1015000
Epoch: 0 Step: 0 LR: 0.00100 Loss 0.6932, Pretrain Micro AUROC 0.5033 Pretrain Micro AUPRC 0.5129 Pretrain Macro AUROC 0.5290 Pretrain Macro AUPRC 0.6243
Epoch: 0 Step: 20 LR: 0.00100 Loss 0.6755, Pretrain Micro AUROC 0.6183 Pretrain Micro AUPRC 0.6046 Pretrain Macro AUROC 0.6265 Pretrain Macro AUPRC 0.6935
Epoch: 0 Step: 40 LR: 0.00100 Loss 0.6658, Pretrain Micro AUROC 0.6217 Pretrain Micro AUPRC 0.6100 Pretrain Macro AUROC 0.6173 Pretrain Macro AUPRC 0.6887
Epoch: 0 Step: 60 LR: 0.00100 Loss 0.6666, Pretrain Micro AUROC 0.6206 Pretrain Micro AUPRC 0.5933 Pretrain Macro AUROC 0.6020 Pretrain Macro AUPRC 0.6596
Epoch: 0 Step: 80 LR: 0.00100 Loss 0.6570, Pretrain Micro AUROC 0.6342 Pretrain Micro AUPRC 0.6104 Pretrain Macro AUROC 0.6116 Pretrain Macro AUPRC 0.6850
Epoch: 0 Step: 100 LR: 0.00100 Loss 0.6507, Pretrain Micro AUROC 0.6551 Pretrain Micro AUPRC 0.6215 Pretrain Macro AUROC 0.6340 Pretrain Macr

In [3]:
## here as a demo, the n_epoch is set to 30. Change it to n_epoch = 500 when you use it
TxGNN.finetune(n_epoch = 30, 
               learning_rate = 5e-4,
               train_print_per_n = 5,
               valid_per_n = 20)

Epoch: 0 LR: 0.00050 Loss 0.6930, Train Micro AUROC 0.5142 Train Micro AUPRC 0.5215 Train Macro AUROC 0.4910 Train Macro AUPRC 0.4988
----- AUROC Performance in Each Relation -----
('drug', 'contraindication', 'disease'): 0.5359948461975489
('drug', 'indication', 'disease'): 0.5360710828008467
('drug', 'off-label use', 'disease'): 0.48515294753921034
('disease', 'rev_contraindication', 'drug'): 0.512682317734401
('disease', 'rev_indication', 'drug'): 0.4469301307346647
('disease', 'rev_off-label use', 'drug'): 0.429232990805841
----- AUPRC Performance in Each Relation -----
('drug', 'contraindication', 'disease'): 0.5422669674557064
('drug', 'indication', 'disease'): 0.5170656094938979
('drug', 'off-label use', 'disease'): 0.5057940107146683
('disease', 'rev_contraindication', 'drug'): 0.5177203883177365
('disease', 'rev_indication', 'drug'): 0.45492647751307785
('disease', 'rev_off-label use', 'drug'): 0.45514037694304715
----------------------------------------------
Validation.....


In [3]:
#TxGNN.save_model('./model_ckpt')
TxGNN.load_pretrained('./model_ckpt')

In [4]:
TxGNN.train_graphmask(relation = 'indication',
                      learning_rate = 3e-4,
                      allowance = 0.005,
                      epochs_per_layer = 3,
                      penalty_scaling = 1,
                      valid_per_n = 20)

output = TxGNN.retrieve_save_gates('./model_ckpt')
TxGNN.save_graphmask_model('./graphmask_model_ckpt')

gate_hidden_size:  32
Enabling layer 1
Running epoch 0 of GraphMask training. Mean divergence=0.0000, mean penalty=1.9051, bce_update=0.6778, bce_original=0.6778, num_masked_l1=0.0000, num_masked_l2=0.0000
----- validation Result -----
Epoch 0, Mean divergence=0.0000, mean penalty=1.9031, bce_update=0.6864, bce_original=0.6864, num_masked_l1=0.0000, num_masked_l2=0.0000
-------------------------------
Running epoch 1 of GraphMask training. Mean divergence=0.0000, mean penalty=1.9041, bce_update=0.6774, bce_original=0.6774, num_masked_l1=0.0000, num_masked_l2=0.0000
Running epoch 2 of GraphMask training. Mean divergence=0.0000, mean penalty=1.9031, bce_update=0.6771, bce_original=0.6771, num_masked_l1=0.0000, num_masked_l2=0.0000
Enabling layer 0
Running epoch 0 of GraphMask training. Mean divergence=0.0000, mean penalty=1.9020, bce_update=0.6768, bce_original=0.6768, num_masked_l1=0.0000, num_masked_l2=0.0000
----- validation Result -----
Epoch 0, Mean divergence=0.0000, mean penalty=1

In [5]:
from txgnn import TxEval
TxEval = TxEval(model = TxGNN)

In [6]:
# evaluate individual diseases
result = TxEval.eval_disease_centric(disease_idxs = [12661.0, 11318.0], 
                                     relation = 'indication', 
                                     save_result = False)

# evaluate the entire test set
result = TxEval.eval_disease_centric(disease_idxs = 'test_set',
                                     show_plot = False, 
                                     verbose = True, 
                                     save_result = True,
                                     return_raw = False)

  0%|          | 0/2 [00:00<?, ?it/s]

Evaluating relation: contraindication


  0%|          | 0/54 [00:00<?, ?it/s]

---------
AUROC mean:  0.5416762086728527
AUROC std:  0.1836740241939981
---------
---------
AUPRC mean:  0.03876062400855024
AUPRC std:  0.06076509586243648
---------
---------
Accuracy mean:  0.974648837277499
Accuracy std:  0.04358800960119679
---------
---------
Sensitivity mean:  0.0
Sensitivity std:  0.0
---------
---------
Specificity mean:  1.0
Specificity std:  0.0
---------
---------
F1 mean:  0.0
F1 std:  0.0
---------
---------
PPV mean:  nan
PPV std:  nan
---------
---------
NPV mean:  0.974648837277499
NPV std:  0.04358800960119679
---------
---------
FPR mean:  0.0
FPR std:  0.0
---------
---------
FNR mean:  1.0
FNR std:  0.0
---------
---------
FDR mean:  nan
FDR std:  nan
---------
---------
# of Pos mean:  32.01851851851852
# of Pos std:  55.05165612631155
---------
---------
Recall@1% mean:  0.009178035270579653
Recall@1% std:  0.030268120598317228
---------
---------
Recall_Random@1% mean:  0.009230146364742498
Recall_Random@1% std:  0.0023165322210400687
---------

  0%|          | 0/66 [00:00<?, ?it/s]

---------
AUROC mean:  0.6581276332459065
AUROC std:  0.24025449899331205
---------
---------
AUPRC mean:  0.019425094701195876
AUPRC std:  0.032550020713776925
---------
---------
Accuracy mean:  0.9958356468628541
Accuracy std:  0.006893993307483263
---------
---------
Sensitivity mean:  0.0
Sensitivity std:  0.0
---------
---------
Specificity mean:  1.0
Specificity std:  0.0
---------
---------
F1 mean:  0.0
F1 std:  0.0
---------
---------
PPV mean:  nan
PPV std:  nan
---------
---------
NPV mean:  0.9958356468628541
NPV std:  0.006893993307483263
---------
---------
FPR mean:  0.0
FPR std:  0.0
---------
---------
FNR mean:  1.0
FNR std:  0.0
---------
---------
FDR mean:  nan
FDR std:  nan
---------
---------
# of Pos mean:  7.5
# of Pos std:  12.41608194677737
---------
---------
Recall@1% mean:  0.015268114499748681
Recall@1% std:  0.05032741556780804
---------
---------
Recall_Random@1% mean:  0.00977556686621463
Recall_Random@1% std:  0.003136990902549219
---------
---------

  0%|          | 0/35 [00:00<?, ?it/s]

---------
AUROC mean:  0.5559410484316637
AUROC std:  0.19329512393601223
---------
---------
AUPRC mean:  0.027870929062758426
AUPRC std:  0.06531688231207894
---------
---------
Accuracy mean:  0.9931381248151433
Accuracy std:  0.005041650577382425
---------
---------
Sensitivity mean:  0.0
Sensitivity std:  0.0
---------
---------
Specificity mean:  1.0
Specificity std:  0.0
---------
---------
F1 mean:  0.0
F1 std:  0.0
---------
---------
PPV mean:  nan
PPV std:  nan
---------
---------
NPV mean:  0.9931381248151433
NPV std:  0.005041650577382425
---------
---------
FPR mean:  0.0
FPR std:  0.0
---------
---------
FNR mean:  1.0
FNR std:  0.0
---------
---------
FDR mean:  nan
FDR std:  nan
---------
---------
# of Pos mean:  3.3142857142857145
# of Pos std:  2.4351172288757157
---------
---------
Recall@1% mean:  0.014285714285714285
Recall@1% std:  0.05802884574739973
---------
---------
Recall_Random@1% mean:  0.007516666666666667
Recall_Random@1% std:  0.0026247025530645356
--

In [7]:
TxEval.retrieve_disease_idxs_test_set('indication')

array([ 7701., 12661., 11195., 16569.,  6343.,  1479., 12621., 14416.,
       13557., 16501., 13446., 13610.,  7105.,  9716.,  5822., 12320.,
       10023., 14513., 12929.,  5683.,  9950., 14661.,  8395.,  7990.,
        6031.,  6255., 14409.,  9589., 11885.,  4285.,  6377., 13130.,
        7773.,  9741., 13381., 15355.,  4536., 12891.,  9087., 14873.,
        6185.,  1812., 15301.,  6474.,  3381.,  6599., 16965.,  7880.,
       16698.,  6252.,  5764.,  4192.,  4467., 13107., 14501., 11318.,
        3452., 11349., 15911., 15100.,  1417.,  4152.,  1319.,  2220.,
       15989.,  8973.])